In [1]:
!pip install -q --no-cache-dir git+https://github.com/nilmtk/nilmtk.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 241.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 230.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 225.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 281.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 271.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0

In [2]:
# ============================================================
# CELL 1 — LOAD PRE-CONVERTED iAWE DATASET
# ============================================================

import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from nilmtk import DataSet

# ------------------------------------------------------------
# Path to your already-converted iawe.h5 (read-only Kaggle input)
# ------------------------------------------------------------
H5_OUTPUT_PATH = "/kaggle/input/datasets/harmanjeetkaur12/iawe-dataset/iawe.h5"

ds = DataSet(H5_OUTPUT_PATH)

print("iAWE dataset loaded successfully.")
print("Available buildings:", list(ds.buildings.keys()))

# ------------------------------------------------------------
# Sanity check: confirm the expected meter structure is present
# ------------------------------------------------------------
building_id = list(ds.buildings.keys())[0]
elec = ds.buildings[building_id].elec

print("\nMains meters:")
print(elec.mains())

print("\nAll meter instance numbers:")
print(sorted(m.instance() for m in elec.meters))

iAWE dataset loaded successfully.
Available buildings: [1]

Mains meters:
MeterGroup(meters=
  ElecMeter(instance=1, building=1, dataset='iAWE', site_meter, appliances=[])
  ElecMeter(instance=2, building=1, dataset='iAWE', site_meter, appliances=[])
)

All meter instance numbers:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [3]:
# ============================================================
# CELL 2 — DEFINE FUNDAMENTAL CLEANING CONFIGURATION
# ============================================================

# iAWE has a single building
HOUSE_IDS = [1]

# Working sampling interval — SAME as UK-DALE, so both datasets
# land on an identical time grid for any downstream comparison
SAMPLE_PERIOD = 6  # seconds

# Power measurement configuration — identical to UK-DALE
PHYSICAL_QUANTITY = "power"
AC_TYPE = "active"

# ------------------------------------------------------------
# Candidate appliance meters
# (single source of truth — replaces the old METER_MAP /
#  appliance_names dicts that disagreed with each other)
# ------------------------------------------------------------
APPLIANCE_METERS = {
    1: {
        "fridge": 3,
        "air_conditioner_1": 4,
        "air_conditioner_2": 5,
        "washing_machine": 6,
        "laptop_computer": 7,
        "iron": 8,
        "kitchen_outlets": 9,
        "television": 10,
        "water_filter": 11,
        "water_motor": 12,
    }
}

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------
print("Fundamental Cleaning Configuration")
print("=" * 55)

print(f"Houses selected: {HOUSE_IDS}")
print(f"Sampling period: {SAMPLE_PERIOD} seconds")
print(f"Physical quantity: {PHYSICAL_QUANTITY}")
print(f"AC type: {AC_TYPE}")

print("\nCandidate appliances:")

for house_id in HOUSE_IDS:
    print(f"\nHouse {house_id}:")

    for appliance, meter_id in APPLIANCE_METERS[house_id].items():
        print(f"  {appliance}: meter {meter_id}")

Fundamental Cleaning Configuration
Houses selected: [1]
Sampling period: 6 seconds
Physical quantity: power
AC type: active

Candidate appliances:

House 1:
  fridge: meter 3
  air_conditioner_1: meter 4
  air_conditioner_2: meter 5
  washing_machine: meter 6
  laptop_computer: meter 7
  iron: meter 8
  kitchen_outlets: meter 9
  television: meter 10
  water_filter: meter 11
  water_motor: meter 12


In [4]:
# ============================================================
# CELL 3 — LOAD RAW SELECTED SIGNALS
# ============================================================

raw_data = {}

for house_id in HOUSE_IDS:
    print(f"\nLoading House {house_id}...")

    house = ds.buildings[house_id]

    # --------------------------------------------------------
    # Load active mains
    # (iAWE has two mains meters — one per phase. nilmtk's
    #  MeterGroup.load() sums across all mains meters in the
    #  group, exactly as it does for any multi-channel mains
    #  setup in UK-DALE, so no manual summation is needed and
    #  no fill_value=0 imputation is introduced.)
    # --------------------------------------------------------
    mains_df = next(
        house.elec.mains().load(
            physical_quantity=PHYSICAL_QUANTITY,
            ac_type=AC_TYPE,
            sample_period=SAMPLE_PERIOD
        )
    )

    mains_series = mains_df.iloc[:, 0]

    raw_data[house_id] = {
        "mains": mains_series
    }

    print(f"  Mains loaded: {len(mains_series):,} samples")

    # --------------------------------------------------------
    # Load candidate appliances
    # --------------------------------------------------------
    for appliance, meter_id in APPLIANCE_METERS[house_id].items():

        appliance_df = next(
            house.elec[meter_id].load(
                physical_quantity=PHYSICAL_QUANTITY,
                ac_type=AC_TYPE,
                sample_period=SAMPLE_PERIOD
            )
        )

        appliance_series = appliance_df.iloc[:, 0]

        raw_data[house_id][appliance] = appliance_series

        print(
            f"  {appliance:<18} "
            f"meter {meter_id:<3} "
            f"{len(appliance_series):,} samples"
        )

print("\nRaw signal loading completed.")


Loading House 1...
Loading data for meter ElecMeterID(instance=2, building=1, dataset='iAWE')     
Done loading data all meters for this chunk.
  Mains loaded: 1,060,533 samples
  fridge             meter 3   859,521 samples
  air_conditioner_1  meter 4   1,476,062 samples
  air_conditioner_2  meter 5   1,410,055 samples
  washing_machine    meter 6   795,280 samples
  laptop_computer    meter 7   858,875 samples
  iron               meter 8   833,570 samples
  kitchen_outlets    meter 9   593,261 samples
  television         meter 10  781,929 samples
  water_filter       meter 11  349,740 samples
  water_motor        meter 12  520,102 samples

Raw signal loading completed.


In [5]:
# ============================================================
# CELL 4 — RAW DATA QUALITY SUMMARY
# ============================================================

def signal_quality_summary(series):
    """
    Return basic quality statistics for a time-series signal.
    No data is modified. Identical to the UK-DALE version.
    """
    index = series.index
    values = series.to_numpy()

    intervals = index.to_series().diff().dropna()
    interval_counts = intervals.value_counts()

    if len(interval_counts) > 0:
        most_common_interval = interval_counts.index[0]
        irregular_intervals = (intervals != most_common_interval).sum()
    else:
        most_common_interval = None
        irregular_intervals = 0

    return {
        "samples": len(series),
        "start": index.min(),
        "end": index.max(),
        "NaN": series.isna().sum(),
        "Inf": np.isinf(values).sum(),
        "negative": (series < 0).sum(),
        "zero": (series == 0).sum(),
        "min_W": series.min(),
        "max_W": series.max(),
        "common_interval": most_common_interval,
        "irregular_intervals": irregular_intervals
    }


quality_rows = []

for house_id in HOUSE_IDS:
    for signal_name, series in raw_data[house_id].items():
        stats = signal_quality_summary(series)
        stats["house"] = house_id
        stats["signal"] = signal_name
        quality_rows.append(stats)

quality_df = pd.DataFrame(quality_rows)

quality_df = quality_df[
    [
        "house", "signal", "samples", "start", "end",
        "NaN", "Inf", "negative", "zero",
        "min_W", "max_W", "common_interval", "irregular_intervals"
    ]
]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(quality_df)

,house,signal,samples,start,end,NaN,Inf,negative,zero,min_W,max_W,common_interval,irregular_intervals
0,1,mains,1060533,2013-05-24 05:30:00+05:30,2013-08-05 21:03:12+05:30,61074,0,0,0,15.331000,8626.443359,0 days 00:00:06,0
1,1,fridge,859521,2013-06-07 05:30:00+05:30,2013-08-05 22:02:00+05:30,30453,0,0,0,0.007000,342.716675,0 days 00:00:06,0
2,1,air_conditioner_1,1476062,2013-06-07 20:34:48+05:30,2013-09-18 08:40:54+05:30,1364448,0,0,0,0.076000,2059.127930,0 days 00:00:06,0
3,1,air_conditioner_2,1410055,2013-06-07 20:35:18+05:30,2013-09-13 18:40:42+05:30,1312814,0,0,0,0.042000,2278.744141,0 days 00:00:06,0
4,1,washing_machine,795280,2013-06-10 07:02:00+05:30,2013-08-04 12:29:54+05:30,789056,0,0,0,0.109000,559.900146,0 days 00:00:06,0
5,1,laptop_computer,858875,2013-06-07 06:34:36+05:30,2013-08-05 22:02:00+05:30,417754,0,0,0,0.002000,105.292168,0 days 00:00:06,0
6,1,iron,833570,2013-06-07 15:07:42+05:30,2013-08-04 12:24:36+05:30,831234,0,0,0,0.101000,1279.634644,0 days 00:00:06,0
7,1,kitchen_outlets,593261,2013-06-24 10:06:06+05:30,2013-08-04 14:52:06+05:30,590173,0,0,0,0.061000,1378.155640,0 days 00:00:06,0
8,1,television,781929,2013-06-12 11:56:24+05:30,2013-08-05 19:09:12+05:30,687949,0,0,0,52.207996,79.218330,0 days 00:00:06,0
9,1,water_filter,349740,2013-07-12 15:08:06+05:30,2013-08-05 22:02:00+05:30,6856,0,0,0,0.103000,56.769669,0 days 00:00:06,0


In [6]:
# ============================================================
# CELL 5 — iAWE-SPECIFIC SENSOR-OUTLIER FLAGGING (documented deviation)
# ============================================================
# UK-DALE has no equivalent step. The iAWE water_motor channel
# (meter 12) contains known sensor-fault spikes around 32 kW on
# a load that never actually draws more than a few hundred watts.
# These are treated as MISSING (NaN), not clipped or zeroed, so
# they flow through the exact same gap-classification /
# interpolation pipeline as any other missing reading below —
# this is the only place iAWE's pipeline differs from UK-DALE's,
# and it is a data-quality fix, not a change in cleaning
# methodology.

OUTLIER_RULES = {
    "water_motor": {"max_plausible_W": 5000}
}

outlier_flagged_data = {}
outlier_report_rows = []

for house_id in HOUSE_IDS:

    outlier_flagged_data[house_id] = {}

    for signal_name, series in raw_data[house_id].items():

        flagged = series.copy()

        if signal_name in OUTLIER_RULES:
            threshold = OUTLIER_RULES[signal_name]["max_plausible_W"]
            bad_mask = flagged > threshold
            n_flagged = int(bad_mask.sum())

            flagged.loc[bad_mask] = np.nan

            outlier_report_rows.append({
                "house": house_id,
                "signal": signal_name,
                "threshold_W": threshold,
                "values_flagged_as_NaN": n_flagged
            })

        outlier_flagged_data[house_id][signal_name] = flagged

outlier_report_df = pd.DataFrame(outlier_report_rows)

print("iAWE SENSOR-OUTLIER FLAGGING")
print("=" * 70)

if len(outlier_report_df) > 0:
    display(outlier_report_df)
else:
    print("No dataset-specific outlier rules triggered.")

iAWE SENSOR-OUTLIER FLAGGING


,house,signal,threshold_W,values_flagged_as_NaN
0,1,water_motor,5000,3


In [7]:
# ============================================================
# CELL 6 — CHECK COMMON TIME OVERLAP
# ============================================================

overlap_rows = []

for house_id in HOUSE_IDS:

    mains = outlier_flagged_data[house_id]["mains"]

    print(f"\nHouse {house_id}")
    print("=" * 60)

    for appliance in APPLIANCE_METERS[house_id]:

        appliance_series = outlier_flagged_data[house_id][appliance]

        overlap_start = max(mains.index.min(), appliance_series.index.min())
        overlap_end = min(mains.index.max(), appliance_series.index.max())

        common_index = mains.index.intersection(appliance_series.index)
        common_index = common_index[
            (common_index >= overlap_start) & (common_index <= overlap_end)
        ]

        row = {
            "house": house_id,
            "appliance": appliance,
            "overlap_start": overlap_start,
            "overlap_end": overlap_end,
            "common_samples": len(common_index),
            "mains_samples": len(mains.loc[common_index]),
            "appliance_samples": len(appliance_series.loc[common_index])
        }

        overlap_rows.append(row)

        print(
            f"{appliance:<20} | "
            f"{overlap_start} \u2192 {overlap_end} | "
            f"{len(common_index):,} common samples"
        )

overlap_df = pd.DataFrame(overlap_rows)

print("\n")
print("OVERLAP SUMMARY")
print("=" * 80)

display(overlap_df)


House 1
fridge               | 2013-06-07 05:30:00+05:30 → 2013-08-05 21:03:12+05:30 | 858,933 common samples
air_conditioner_1    | 2013-06-07 20:34:48+05:30 → 2013-08-05 21:03:12+05:30 | 849,885 common samples
air_conditioner_2    | 2013-06-07 20:35:18+05:30 → 2013-08-05 21:03:12+05:30 | 849,880 common samples
washing_machine      | 2013-06-10 07:02:00+05:30 → 2013-08-04 12:29:54+05:30 | 795,280 common samples
laptop_computer      | 2013-06-07 06:34:36+05:30 → 2013-08-05 21:03:12+05:30 | 858,287 common samples
iron                 | 2013-06-07 15:07:42+05:30 → 2013-08-04 12:24:36+05:30 | 833,570 common samples
kitchen_outlets      | 2013-06-24 10:06:06+05:30 → 2013-08-04 14:52:06+05:30 | 593,261 common samples
television           | 2013-06-12 11:56:24+05:30 → 2013-08-05 19:09:12+05:30 | 781,929 common samples
water_filter         | 2013-07-12 15:08:06+05:30 → 2013-08-05 21:03:12+05:30 | 349,152 common samples
water_motor          | 2013-05-30 12:51:36+05:30 → 2013-07-05 15:41:42+05

,house,appliance,overlap_start,overlap_end,common_samples,mains_samples,appliance_samples
0,1,fridge,2013-06-07 05:30:00+05:30,2013-08-05 21:03:12+05:30,858933,858933,858933
1,1,air_conditioner_1,2013-06-07 20:34:48+05:30,2013-08-05 21:03:12+05:30,849885,849885,849885
2,1,air_conditioner_2,2013-06-07 20:35:18+05:30,2013-08-05 21:03:12+05:30,849880,849880,849880
3,1,washing_machine,2013-06-10 07:02:00+05:30,2013-08-04 12:29:54+05:30,795280,795280,795280
4,1,laptop_computer,2013-06-07 06:34:36+05:30,2013-08-05 21:03:12+05:30,858287,858287,858287
5,1,iron,2013-06-07 15:07:42+05:30,2013-08-04 12:24:36+05:30,833570,833570,833570
6,1,kitchen_outlets,2013-06-24 10:06:06+05:30,2013-08-04 14:52:06+05:30,593261,593261,593261
7,1,television,2013-06-12 11:56:24+05:30,2013-08-05 19:09:12+05:30,781929,781929,781929
8,1,water_filter,2013-07-12 15:08:06+05:30,2013-08-05 21:03:12+05:30,349152,349152,349152
9,1,water_motor,2013-05-30 12:51:36+05:30,2013-07-05 15:41:42+05:30,520102,520102,520102


In [8]:
# ============================================================
# CELL 7 — ANALYZE MISSING-VALUE ALIGNMENT
# ============================================================

missing_rows = []

for house_id in HOUSE_IDS:

    mains = outlier_flagged_data[house_id]["mains"]

    print(f"\nHouse {house_id}")
    print("=" * 70)

    for appliance in APPLIANCE_METERS[house_id]:

        appliance_series = outlier_flagged_data[house_id][appliance]

        common_index = mains.index.intersection(appliance_series.index)

        mains_common = mains.loc[common_index]
        appliance_common = appliance_series.loc[common_index]

        mains_nan = mains_common.isna()
        appliance_nan = appliance_common.isna()

        both_nan = mains_nan & appliance_nan
        mains_only_nan = mains_nan & ~appliance_nan
        appliance_only_nan = ~mains_nan & appliance_nan
        neither_nan = ~mains_nan & ~appliance_nan

        row = {
            "house": house_id,
            "appliance": appliance,
            "common_samples": len(common_index),
            "mains_nan": mains_nan.sum(),
            "appliance_nan": appliance_nan.sum(),
            "both_nan": both_nan.sum(),
            "mains_only_nan": mains_only_nan.sum(),
            "appliance_only_nan": appliance_only_nan.sum(),
            "both_valid": neither_nan.sum()
        }

        missing_rows.append(row)

        print(
            f"{appliance:<20} | "
            f"Mains NaN: {mains_nan.sum():>9,} | "
            f"Appliance NaN: {appliance_nan.sum():>9,} | "
            f"Both NaN: {both_nan.sum():>9,} | "
            f"Both valid: {neither_nan.sum():>9,}"
        )

missing_df = pd.DataFrame(missing_rows)

print("\n")
print("MISSING-VALUE ALIGNMENT SUMMARY")
print("=" * 90)

display(missing_df)


House 1
fridge               | Mains NaN:    35,397 | Appliance NaN:    30,453 | Both NaN:    17,519 | Both valid:   810,602
air_conditioner_1    | Mains NaN:    34,837 | Appliance NaN:   776,587 | Both NaN:    34,176 | Both valid:    72,637
air_conditioner_2    | Mains NaN:    34,837 | Appliance NaN:   756,702 | Both NaN:    32,562 | Both valid:    90,903
washing_machine      | Mains NaN:    29,918 | Appliance NaN:   789,056 | Both NaN:    29,918 | Both valid:     6,224
laptop_computer      | Mains NaN:    35,127 | Appliance NaN:   417,487 | Both NaN:    21,811 | Both valid:   427,484
iron                 | Mains NaN:    34,837 | Appliance NaN:   831,234 | Both NaN:    34,837 | Both valid:     2,336
kitchen_outlets      | Mains NaN:    17,974 | Appliance NaN:   590,173 | Both NaN:    17,974 | Both valid:     3,088
television           | Mains NaN:    27,383 | Appliance NaN:   687,949 | Both NaN:    24,157 | Both valid:    90,754
water_filter         | Mains NaN:     7,908 | Appliance

,house,appliance,common_samples,mains_nan,appliance_nan,both_nan,mains_only_nan,appliance_only_nan,both_valid
0,1,fridge,858933,35397,30453,17519,17878,12934,810602
1,1,air_conditioner_1,849885,34837,776587,34176,661,742411,72637
2,1,air_conditioner_2,849880,34837,756702,32562,2275,724140,90903
3,1,washing_machine,795280,29918,789056,29918,0,759138,6224
4,1,laptop_computer,858287,35127,417487,21811,13316,395676,427484
5,1,iron,833570,34837,831234,34837,0,796397,2336
6,1,kitchen_outlets,593261,17974,590173,17974,0,572199,3088
7,1,television,781929,27383,687949,24157,3226,663792,90754
8,1,water_filter,349152,7908,6856,1699,6209,5157,336087
9,1,water_motor,520102,40193,40349,27575,12618,12774,467135


In [9]:
# ============================================================
# CELL 8 — ANALYZE CONSECUTIVE NaN GAPS
# ============================================================

def get_nan_gap_lengths(series):
    """Find lengths of consecutive NaN runs, in samples."""
    is_nan = series.isna()
    groups = (is_nan != is_nan.shift()).cumsum()
    gap_lengths = is_nan.groupby(groups).sum()
    gap_lengths = gap_lengths[gap_lengths > 0]
    return gap_lengths.astype(int)


gap_rows = []

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("=" * 80)

    for signal_name, series in outlier_flagged_data[house_id].items():

        gap_lengths = get_nan_gap_lengths(series)

        if len(gap_lengths) == 0:
            print(f"{signal_name:<20} No NaN gaps")
            continue

        gap_seconds = gap_lengths * SAMPLE_PERIOD

        row = {
            "house": house_id,
            "signal": signal_name,
            "number_of_nan_gaps": len(gap_lengths),
            "shortest_gap_samples": gap_lengths.min(),
            "median_gap_samples": gap_lengths.median(),
            "longest_gap_samples": gap_lengths.max(),
            "shortest_gap_seconds": gap_seconds.min(),
            "median_gap_seconds": gap_seconds.median(),
            "longest_gap_seconds": gap_seconds.max()
        }

        gap_rows.append(row)

        print(
            f"{signal_name:<20} "
            f"gaps: {len(gap_lengths):>6,} | "
            f"median: {gap_seconds.median():>8.0f} sec | "
            f"longest: {gap_seconds.max():>10.0f} sec"
        )

gap_df = pd.DataFrame(gap_rows)

print("\n")
print("NaN GAP SUMMARY")
print("=" * 100)

display(gap_df)


House 1
mains                gaps:    104 | median:     2001 sec | longest:      30876 sec
fridge               gaps:     80 | median:      963 sec | longest:      12090 sec
air_conditioner_1    gaps:     75 | median:    78126 sec | longest:    3338682 sec
air_conditioner_2    gaps:    105 | median:    55248 sec | longest:    3253632 sec
washing_machine      gaps:     26 | median:   255123 sec | longest:     414666 sec
laptop_computer      gaps:    207 | median:     3678 sec | longest:      90990 sec
iron                 gaps:     12 | median:   501432 sec | longest:     792504 sec
kitchen_outlets      gaps:     10 | median:    63798 sec | longest:    1825950 sec
television           gaps:    207 | median:     4740 sec | longest:    1476972 sec
water_filter         gaps:     32 | median:      564 sec | longest:       6786 sec
water_motor          gaps:     52 | median:     2970 sec | longest:      38460 sec


NaN GAP SUMMARY


,house,signal,number_of_nan_gaps,shortest_gap_samples,median_gap_samples,longest_gap_samples,shortest_gap_seconds,median_gap_seconds,longest_gap_seconds
0,1,mains,104,3,333.5,5146,18,2001.0,30876
1,1,fridge,80,2,160.5,2015,12,963.0,12090
2,1,air_conditioner_1,75,2,13021.0,556447,12,78126.0,3338682
3,1,air_conditioner_2,105,17,9208.0,542272,102,55248.0,3253632
4,1,washing_machine,26,82,42520.5,69111,492,255123.0,414666
5,1,laptop_computer,207,1,613.0,15165,6,3678.0,90990
6,1,iron,12,2,83572.0,132084,12,501432.0,792504
7,1,kitchen_outlets,10,45,10633.0,304325,270,63798.0,1825950
8,1,television,207,2,790.0,246162,12,4740.0,1476972
9,1,water_filter,32,4,94.0,1131,24,564.0,6786


In [10]:
# ============================================================
# CELL 9 — CLASSIFY NaN GAPS BY DURATION
# ============================================================

SHORT_GAP_SECONDS = 300  # 5 minutes — same threshold as UK-DALE

classification_rows = []

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("=" * 90)

    for signal_name, series in outlier_flagged_data[house_id].items():

        gap_lengths = get_nan_gap_lengths(series)

        if len(gap_lengths) == 0:
            continue

        gap_seconds = gap_lengths * SAMPLE_PERIOD

        short_gaps = gap_seconds[gap_seconds <= SHORT_GAP_SECONDS]
        long_gaps = gap_seconds[gap_seconds > SHORT_GAP_SECONDS]

        short_samples = gap_lengths[gap_seconds <= SHORT_GAP_SECONDS].sum()
        long_samples = gap_lengths[gap_seconds > SHORT_GAP_SECONDS].sum()

        total_nan = series.isna().sum()

        row = {
            "house": house_id,
            "signal": signal_name,
            "total_nan_samples": total_nan,
            "short_gap_count": len(short_gaps),
            "short_gap_samples": int(short_samples),
            "short_gap_seconds": int(short_samples * SAMPLE_PERIOD),
            "long_gap_count": len(long_gaps),
            "long_gap_samples": int(long_samples),
            "long_gap_seconds": int(long_samples * SAMPLE_PERIOD),
            "short_gap_%_of_nan": (
                short_samples / total_nan * 100 if total_nan > 0 else 0
            ),
            "long_gap_%_of_nan": (
                long_samples / total_nan * 100 if total_nan > 0 else 0
            )
        }

        classification_rows.append(row)

        print(
            f"{signal_name:<20} | "
            f"NaNs: {total_nan:>10,} | "
            f"short gaps: {len(short_gaps):>4} "
            f"({short_samples:>10,} samples) | "
            f"long gaps: {len(long_gaps):>4} "
            f"({long_samples:>10,} samples)"
        )

gap_classification_df = pd.DataFrame(classification_rows)

print("\n")
print("NaN GAP CLASSIFICATION SUMMARY")
print("=" * 110)

display(gap_classification_df)


House 1
mains                | NaNs:     61,074 | short gaps:    7 (       151 samples) | long gaps:   97 (    60,923 samples)
fridge               | NaNs:     30,453 | short gaps:   18 (       301 samples) | long gaps:   62 (    30,152 samples)
air_conditioner_1    | NaNs:  1,364,448 | short gaps:    4 (        58 samples) | long gaps:   71 ( 1,364,390 samples)
air_conditioner_2    | NaNs:  1,312,814 | short gaps:    6 (       148 samples) | long gaps:   99 ( 1,312,666 samples)
washing_machine      | NaNs:    789,056 | short gaps:    0 (         0 samples) | long gaps:   26 (   789,056 samples)
laptop_computer      | NaNs:    417,754 | short gaps:   12 (       268 samples) | long gaps:  195 (   417,486 samples)
iron                 | NaNs:    831,234 | short gaps:    1 (         2 samples) | long gaps:   11 (   831,232 samples)
kitchen_outlets      | NaNs:    590,173 | short gaps:    1 (        45 samples) | long gaps:    9 (   590,128 samples)
television           | NaNs:    687,949

,house,signal,total_nan_samples,short_gap_count,short_gap_samples,short_gap_seconds,long_gap_count,long_gap_samples,long_gap_seconds,short_gap_%_of_nan,long_gap_%_of_nan
0,1,mains,61074,7,151,906,97,60923,365538,0.247241,99.752759
1,1,fridge,30453,18,301,1806,62,30152,180912,0.988408,99.011592
2,1,air_conditioner_1,1364448,4,58,348,71,1364390,8186340,0.004251,99.995749
3,1,air_conditioner_2,1312814,6,148,888,99,1312666,7875996,0.011273,99.988727
4,1,washing_machine,789056,0,0,0,26,789056,4734336,0.000000,100.000000
5,1,laptop_computer,417754,12,268,1608,195,417486,2504916,0.064153,99.935847
6,1,iron,831234,1,2,12,11,831232,4987392,0.000241,99.999759
7,1,kitchen_outlets,590173,1,45,270,9,590128,3540768,0.007625,99.992375
8,1,television,687949,16,395,2370,191,687554,4125324,0.057417,99.942583
9,1,water_filter,6856,13,275,1650,19,6581,39486,4.011085,95.988915


In [11]:
# ============================================================
# CELL 10 — INSPECT LOCATIONS OF SHORT NaN GAPS
# ============================================================

SHORT_GAP_SAMPLES = SHORT_GAP_SECONDS // SAMPLE_PERIOD


def get_nan_gap_info(series):
    """Return information about every consecutive NaN gap."""
    is_nan = series.isna()
    groups = (is_nan != is_nan.shift()).cumsum()

    gap_info = []

    for group_id, group in series.groupby(groups):
        if not group.isna().all():
            continue

        gap_length = len(group)

        gap_info.append({
            "start": group.index.min(),
            "end": group.index.max(),
            "samples": gap_length,
            "seconds": gap_length * SAMPLE_PERIOD
        })

    return pd.DataFrame(gap_info)


short_gap_rows = []

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("=" * 90)

    for signal_name, series in outlier_flagged_data[house_id].items():

        gap_info = get_nan_gap_info(series)

        if gap_info.empty:
            continue

        short_gaps = gap_info[gap_info["seconds"] <= SHORT_GAP_SECONDS].copy()

        if short_gaps.empty:
            print(f"{signal_name:<20} No short gaps")
            continue

        isolated_count = 0
        edge_count = 0

        for _, gap in short_gaps.iterrows():

            gap_start = gap["start"]
            gap_end = gap["end"]

            try:
                position_start = series.index.get_loc(gap_start)
                position_end = series.index.get_loc(gap_end)

                has_previous = position_start > 0
                has_next = position_end < len(series) - 1

                previous_valid = (
                    has_previous and not pd.isna(series.iloc[position_start - 1])
                )
                next_valid = (
                    has_next and not pd.isna(series.iloc[position_end + 1])
                )

                if previous_valid and next_valid:
                    isolated_count += 1
                else:
                    edge_count += 1

            except Exception:
                edge_count += 1

        short_gap_rows.append({
            "house": house_id,
            "signal": signal_name,
            "short_gap_count": len(short_gaps),
            "isolated_short_gaps": isolated_count,
            "edge_or_unbounded_gaps": edge_count,
            "short_gap_samples": int(short_gaps["samples"].sum()),
            "short_gap_seconds": int(short_gaps["seconds"].sum())
        })

        print(
            f"{signal_name:<20} | "
            f"short gaps: {len(short_gaps):>4} | "
            f"isolated: {isolated_count:>4} | "
            f"edge/unbounded: {edge_count:>4}"
        )

short_gap_df = pd.DataFrame(short_gap_rows)

print("\n")
print("SHORT GAP LOCATION SUMMARY")
print("=" * 100)

display(short_gap_df)


House 1
mains                | short gaps:    7 | isolated:    7 | edge/unbounded:    0
fridge               | short gaps:   18 | isolated:   18 | edge/unbounded:    0
air_conditioner_1    | short gaps:    4 | isolated:    4 | edge/unbounded:    0
air_conditioner_2    | short gaps:    6 | isolated:    6 | edge/unbounded:    0
washing_machine      No short gaps
laptop_computer      | short gaps:   12 | isolated:   12 | edge/unbounded:    0
iron                 | short gaps:    1 | isolated:    1 | edge/unbounded:    0
kitchen_outlets      | short gaps:    1 | isolated:    1 | edge/unbounded:    0
television           | short gaps:   16 | isolated:   16 | edge/unbounded:    0
water_filter         | short gaps:   13 | isolated:   13 | edge/unbounded:    0
water_motor          | short gaps:    4 | isolated:    4 | edge/unbounded:    0


SHORT GAP LOCATION SUMMARY


,house,signal,short_gap_count,isolated_short_gaps,edge_or_unbounded_gaps,short_gap_samples,short_gap_seconds
0,1,mains,7,7,0,151,906
1,1,fridge,18,18,0,301,1806
2,1,air_conditioner_1,4,4,0,58,348
3,1,air_conditioner_2,6,6,0,148,888
4,1,laptop_computer,12,12,0,268,1608
5,1,iron,1,1,0,2,12
6,1,kitchen_outlets,1,1,0,45,270
7,1,television,16,16,0,395,2370
8,1,water_filter,13,13,0,275,1650
9,1,water_motor,4,4,0,43,258


In [12]:
# ============================================================
# CELL 11 — FUNDAMENTAL DATA CLEANING
# ============================================================

MAX_INTERPOLATION_SECONDS = 300  # 5 minutes — same as UK-DALE
MAX_INTERPOLATION_SAMPLES = MAX_INTERPOLATION_SECONDS // SAMPLE_PERIOD

print("Starting fundamental data cleaning...")
print("=" * 70)

cleaned_data = {}


def interpolate_only_short_gaps(series, max_gap_samples):
    """
    Linearly interpolate only complete NaN gaps whose length
    is <= max_gap_samples. Longer NaN gaps are left untouched.
    Identical logic to the UK-DALE version.
    """
    cleaned = series.copy()
    is_nan = cleaned.isna()
    groups = (is_nan != is_nan.shift()).cumsum()

    for group_id, group in cleaned.groupby(groups):

        if not group.isna().all():
            continue

        gap_length = len(group)

        if gap_length <= max_gap_samples:

            start_position = cleaned.index.get_loc(group.index[0])
            end_position = cleaned.index.get_loc(group.index[-1])

            if (
                start_position > 0
                and end_position < len(cleaned) - 1
                and not pd.isna(cleaned.iloc[start_position - 1])
                and not pd.isna(cleaned.iloc[end_position + 1])
            ):

                previous_value = cleaned.iloc[start_position - 1]
                next_value = cleaned.iloc[end_position + 1]

                gap_positions = range(start_position, end_position + 1)
                number_of_missing = gap_length

                for i, position in enumerate(gap_positions, start=1):
                    fraction = i / (number_of_missing + 1)
                    cleaned.iloc[position] = (
                        previous_value + fraction * (next_value - previous_value)
                    )

    return cleaned


for house_id in HOUSE_IDS:

    print(f"\nCleaning House {house_id}")
    print("-" * 70)

    cleaned_data[house_id] = {}

    for signal_name, series in outlier_flagged_data[house_id].items():

        cleaned_series = interpolate_only_short_gaps(
            series, MAX_INTERPOLATION_SAMPLES
        )

        cleaned_data[house_id][signal_name] = cleaned_series

        original_nan = series.isna().sum()
        remaining_nan = cleaned_series.isna().sum()
        interpolated = original_nan - remaining_nan

        print(
            f"{signal_name:<20} | "
            f"original NaN: {original_nan:>10,} | "
            f"interpolated: {interpolated:>10,} | "
            f"remaining NaN: {remaining_nan:>10,}"
        )

print("\n")
print("Fundamental cleaning completed.")

Starting fundamental data cleaning...

Cleaning House 1
----------------------------------------------------------------------
mains                | original NaN:     61,074 | interpolated:        151 | remaining NaN:     60,923
fridge               | original NaN:     30,453 | interpolated:        301 | remaining NaN:     30,152
air_conditioner_1    | original NaN:  1,364,448 | interpolated:         58 | remaining NaN:  1,364,390
air_conditioner_2    | original NaN:  1,312,814 | interpolated:        148 | remaining NaN:  1,312,666
washing_machine      | original NaN:    789,056 | interpolated:          0 | remaining NaN:    789,056
laptop_computer      | original NaN:    417,754 | interpolated:        268 | remaining NaN:    417,486
iron                 | original NaN:    831,234 | interpolated:          2 | remaining NaN:    831,232
kitchen_outlets      | original NaN:    590,173 | interpolated:         45 | remaining NaN:    590,128
television           | original NaN:    687,949 |

In [13]:
# ============================================================
# CELL 12 — VERIFY FUNDAMENTAL CLEANING
# ============================================================

verification_rows = []

for house_id in HOUSE_IDS:

    for signal_name, series in cleaned_data[house_id].items():

        gap_lengths = get_nan_gap_lengths(series)

        if len(gap_lengths) > 0:
            gap_seconds = gap_lengths * SAMPLE_PERIOD
            remaining_short_gaps = (
                gap_seconds <= MAX_INTERPOLATION_SECONDS
            ).sum()
            longest_gap_seconds = gap_seconds.max()
        else:
            remaining_short_gaps = 0
            longest_gap_seconds = 0

        # NOTE: negative values are only counted here, never
        # clipped to zero — same philosophy as UK-DALE. Clipping
        # (if desired) is a modeling-stage decision, not a
        # cleaning-stage one, and belongs downstream of this file.
        negative_values = (series < 0).sum()
        infinite_values = np.isinf(series.to_numpy()).sum()
        zero_values = (series == 0).sum()

        verification_rows.append({
            "house": house_id,
            "signal": signal_name,
            "remaining_NaN": series.isna().sum(),
            "remaining_short_gaps": remaining_short_gaps,
            "longest_remaining_gap_sec": longest_gap_seconds,
            "negative_values": negative_values,
            "infinite_values": infinite_values,
            "zero_values": zero_values
        })

verification_df = pd.DataFrame(verification_rows)

print("FUNDAMENTAL CLEANING VERIFICATION")
print("=" * 100)

display(verification_df)

FUNDAMENTAL CLEANING VERIFICATION


,house,signal,remaining_NaN,remaining_short_gaps,longest_remaining_gap_sec,negative_values,infinite_values,zero_values
0,1,mains,60923,0,30876,0,0,0
1,1,fridge,30152,0,12090,0,0,0
2,1,air_conditioner_1,1364390,0,3338682,0,0,0
3,1,air_conditioner_2,1312666,0,3253632,0,0,0
4,1,washing_machine,789056,0,414666,0,0,0
5,1,laptop_computer,417486,0,90990,0,0,0
6,1,iron,831232,0,792504,0,0,0
7,1,kitchen_outlets,590128,0,1825950,0,0,0
8,1,television,687554,0,1476972,0,0,0
9,1,water_filter,6581,0,6786,0,0,0


In [14]:
# ============================================================
# CELL 13 — CREATE ALIGNED MAINS–APPLIANCE PAIRS
# ============================================================

paired_data = {}
alignment_rows = []

print("Creating aligned mains\u2013appliance pairs...")
print("=" * 100)

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("-" * 100)

    mains = cleaned_data[house_id]["mains"]

    paired_data[house_id] = {}

    for appliance in APPLIANCE_METERS[house_id]:

        appliance_series = cleaned_data[house_id][appliance]

        common_index = mains.index.intersection(appliance_series.index)

        mains_common = mains.loc[common_index]
        appliance_common = appliance_series.loc[common_index]

        valid_mask = mains_common.notna() & appliance_common.notna()

        mains_valid = mains_common.loc[valid_mask]
        appliance_valid = appliance_common.loc[valid_mask]

        assert mains_valid.index.equals(appliance_valid.index)

        pair_df = pd.DataFrame({
            "mains": mains_valid,
            "appliance": appliance_valid
        })

        paired_data[house_id][appliance] = pair_df

        original_common_samples = len(common_index)
        valid_samples = len(pair_df)
        removed_samples = original_common_samples - valid_samples

        alignment_rows.append({
            "house": house_id,
            "appliance": appliance,
            "common_samples_before_NaN_removal": original_common_samples,
            "valid_aligned_samples": valid_samples,
            "removed_due_to_NaN": removed_samples,
            "retained_percentage": valid_samples / original_common_samples * 100
        })

        print(
            f"{appliance:<20} | "
            f"before: {original_common_samples:>10,} | "
            f"valid: {valid_samples:>10,} | "
            f"removed: {removed_samples:>10,} | "
            f"retained: {valid_samples / original_common_samples * 100:>6.2f}%"
        )

alignment_df = pd.DataFrame(alignment_rows)

print("\n")
print("ALIGNMENT SUMMARY")
print("=" * 100)

display(alignment_df)

Creating aligned mains–appliance pairs...

House 1
----------------------------------------------------------------------------------------------------
fridge               | before:    858,933 | valid:    810,940 | removed:     47,993 | retained:  94.41%
air_conditioner_1    | before:    849,885 | valid:     72,695 | removed:    777,190 | retained:   8.55%
air_conditioner_2    | before:    849,880 | valid:     91,051 | removed:    758,829 | retained:  10.71%
washing_machine      | before:    795,280 | valid:      6,224 | removed:    789,056 | retained:   0.78%
laptop_computer      | before:    858,287 | valid:    427,850 | removed:    430,437 | retained:  49.85%
iron                 | before:    833,570 | valid:      2,338 | removed:    831,232 | retained:   0.28%
kitchen_outlets      | before:    593,261 | valid:      3,133 | removed:    590,128 | retained:   0.53%
television           | before:    781,929 | valid:     91,135 | removed:    690,794 | retained:  11.66%
water_filter    

,house,appliance,common_samples_before_NaN_removal,valid_aligned_samples,removed_due_to_NaN,retained_percentage
0,1,fridge,858933,810940,47993,94.412486
1,1,air_conditioner_1,849885,72695,777190,8.553510
2,1,air_conditioner_2,849880,91051,758829,10.713395
3,1,washing_machine,795280,6224,789056,0.782617
4,1,laptop_computer,858287,427850,430437,49.849293
5,1,iron,833570,2338,831232,0.280480
6,1,kitchen_outlets,593261,3133,590128,0.528098
7,1,television,781929,91135,690794,11.655150
8,1,water_filter,349152,336322,12830,96.325383
9,1,water_motor,520102,467244,52858,89.836994


In [15]:
# ============================================================
# CELL 14 — VERIFY ALIGNED DATA AND CONTINUOUS SEGMENTS
# ============================================================

SEGMENT_GAP_SECONDS = SAMPLE_PERIOD


def analyze_continuous_segments(pair_df):
    if len(pair_df) == 0:
        return {
            "segments": 0,
            "irregular_intervals": 0,
            "longest_segment_samples": 0,
            "shortest_segment_samples": 0
        }

    intervals = pair_df.index.to_series().diff().dropna()
    breaks = intervals != pd.Timedelta(seconds=SAMPLE_PERIOD)
    number_of_breaks = breaks.sum()
    segment_ids = breaks.cumsum()
    segment_lengths = pair_df.groupby(segment_ids).size()

    return {
        "segments": len(segment_lengths),
        "irregular_intervals": int(number_of_breaks),
        "longest_segment_samples": int(segment_lengths.max()),
        "shortest_segment_samples": int(segment_lengths.min())
    }


segment_rows = []

print("ALIGNMENT AND CONTINUOUS-SEGMENT VERIFICATION")
print("=" * 110)

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("-" * 110)

    for appliance, pair_df in paired_data[house_id].items():

        assert pair_df.index.is_monotonic_increasing
        assert pair_df.index.is_unique
        assert pair_df["mains"].notna().all()
        assert pair_df["appliance"].notna().all()

        segment_stats = analyze_continuous_segments(pair_df)

        row = {
            "house": house_id,
            "appliance": appliance,
            "samples": len(pair_df),
            "segments": segment_stats["segments"],
            "breaks": segment_stats["irregular_intervals"],
            "longest_segment_samples": segment_stats["longest_segment_samples"],
            "shortest_segment_samples": segment_stats["shortest_segment_samples"],
            "longest_segment_hours": (
                segment_stats["longest_segment_samples"] * SAMPLE_PERIOD / 3600
            )
        }

        segment_rows.append(row)

        print(
            f"{appliance:<20} | "
            f"samples: {len(pair_df):>10,} | "
            f"segments: {segment_stats['segments']:>5,} | "
            f"breaks: {segment_stats['irregular_intervals']:>5,} | "
            f"longest: {row['longest_segment_hours']:>8.2f} h"
        )

segment_df = pd.DataFrame(segment_rows)

print("\n")
print("SEGMENT SUMMARY")
print("=" * 110)

display(segment_df)

ALIGNMENT AND CONTINUOUS-SEGMENT VERIFICATION

House 1
--------------------------------------------------------------------------------------------------------------
fridge               | samples:    810,940 | segments:    86 | breaks:    85 | longest:    76.33 h
air_conditioner_1    | samples:     72,695 | segments:    64 | breaks:    63 | longest:     9.49 h
air_conditioner_2    | samples:     91,051 | segments:    93 | breaks:    92 | longest:     8.28 h
washing_machine      | samples:      6,224 | segments:    27 | breaks:    26 | longest:     0.73 h
laptop_computer      | samples:    427,850 | segments:   206 | breaks:   205 | longest:    17.41 h
iron                 | samples:      2,338 | segments:    12 | breaks:    11 | longest:     0.48 h
kitchen_outlets      | samples:      3,133 | segments:    10 | breaks:     9 | longest:     2.51 h
television           | samples:     91,135 | segments:   188 | breaks:   187 | longest:     3.75 h
water_filter         | samples:    336,322

,house,appliance,samples,segments,breaks,longest_segment_samples,shortest_segment_samples,longest_segment_hours
0,1,fridge,810940,86,85,45795,2,76.325000
1,1,air_conditioner_1,72695,64,63,5696,110,9.493333
2,1,air_conditioner_2,91051,93,92,4967,52,8.278333
3,1,washing_machine,6224,27,26,440,38,0.733333
4,1,laptop_computer,427850,206,205,10446,20,17.410000
5,1,iron,2338,12,11,286,35,0.476667
6,1,kitchen_outlets,3133,10,9,1504,46,2.506667
7,1,television,91135,188,187,2248,5,3.746667
8,1,water_filter,336322,29,28,34985,252,58.308333
9,1,water_motor,467244,61,60,45790,55,76.316667


In [16]:
# ============================================================
# CELL 15 — FINAL FUNDAMENTAL CLEANING VALIDATION
# ============================================================

final_validation_rows = []

for house_id in HOUSE_IDS:
    for appliance, pair_df in paired_data[house_id].items():

        assert pair_df.index.is_monotonic_increasing
        assert pair_df.index.is_unique
        assert pair_df["mains"].notna().all()
        assert pair_df["appliance"].notna().all()

        intervals = pair_df.index.to_series().diff()
        segment_breaks = intervals != pd.Timedelta(seconds=SAMPLE_PERIOD)
        segment_breaks.iloc[0] = False
        segment_ids = segment_breaks.cumsum()

        irregular_intervals = 0
        total_internal_intervals = 0

        for _, segment in pair_df.groupby(segment_ids):

            if len(segment) <= 1:
                continue

            segment_intervals = segment.index.to_series().diff().dropna()
            total_internal_intervals += len(segment_intervals)
            irregular_intervals += (
                segment_intervals != pd.Timedelta(seconds=SAMPLE_PERIOD)
            ).sum()

        exact_period_intervals = total_internal_intervals - irregular_intervals

        mains_negative = (pair_df["mains"] < 0).sum()
        appliance_negative = (pair_df["appliance"] < 0).sum()
        mains_inf = np.isinf(pair_df["mains"].to_numpy()).sum()
        appliance_inf = np.isinf(pair_df["appliance"].to_numpy()).sum()

        number_of_segments = segment_ids.nunique()

        original_samples = len(raw_data[house_id]["mains"])
        retained_samples = len(pair_df)
        retention_percentage = retained_samples / original_samples * 100

        final_validation_rows.append({
            "house": house_id,
            "appliance": appliance,
            "retained_samples": retained_samples,
            "retention_%": retention_percentage,
            "segments": number_of_segments,
            "exact_period_intervals": int(exact_period_intervals),
            "irregular_intervals": int(irregular_intervals),
            "mains_negative": int(mains_negative),
            "appliance_negative": int(appliance_negative),
            "mains_inf": int(mains_inf),
            "appliance_inf": int(appliance_inf)
        })

final_validation_df = pd.DataFrame(final_validation_rows)

print("FINAL FUNDAMENTAL CLEANING VALIDATION")
print("=" * 120)

display(final_validation_df)

print("\nValidation checks:")
print("-" * 70)

print("Maximum irregular intervals within segments:",
      final_validation_df["irregular_intervals"].max())
print("Maximum negative mains values:",
      final_validation_df["mains_negative"].max())
print("Maximum negative appliance values:",
      final_validation_df["appliance_negative"].max())
print("Maximum infinite mains values:",
      final_validation_df["mains_inf"].max())
print("Maximum infinite appliance values:",
      final_validation_df["appliance_inf"].max())

print("\nFundamental data cleaning validation completed.")

FINAL FUNDAMENTAL CLEANING VALIDATION


,house,appliance,retained_samples,retention_%,segments,exact_period_intervals,irregular_intervals,mains_negative,appliance_negative,mains_inf,appliance_inf
0,1,fridge,810940,76.465325,86,810854,0,0,0,0,0
1,1,air_conditioner_1,72695,6.854572,64,72631,0,0,0,0,0
2,1,air_conditioner_2,91051,8.585400,93,90958,0,0,0,0,0
3,1,washing_machine,6224,0.586875,27,6197,0,0,0,0,0
4,1,laptop_computer,427850,40.342922,206,427644,0,0,0,0,0
5,1,iron,2338,0.220455,12,2326,0,0,0,0,0
6,1,kitchen_outlets,3133,0.295417,10,3123,0,0,0,0,0
7,1,television,91135,8.593321,188,90947,0,0,0,0,0
8,1,water_filter,336322,31.712545,29,336293,0,0,0,0,0
9,1,water_motor,467244,44.057469,61,467183,0,0,0,0,0



Validation checks:
----------------------------------------------------------------------
Maximum irregular intervals within segments: 0
Maximum negative mains values: 0
Maximum negative appliance values: 0
Maximum infinite mains values: 0
Maximum infinite appliance values: 0

Fundamental data cleaning validation completed.


In [17]:
# ============================================================
# CELL 16 — SAVE CLEANED DATASET
# ============================================================

import os

PROCESSED_DIR = "kaggle/working/"
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    PROCESSED_DIR,
    "iawe_fundamental_cleaned_6s.h5"
)

with pd.HDFStore(OUTPUT_PATH, mode="w") as store:
    for house_id in paired_data:
        for appliance, pair_df in paired_data[house_id].items():
            key = f"house_{house_id}/{appliance}"
            store.put(key, pair_df, format="table", data_columns=True)

print("Cleaned dataset saved successfully.")
print("File:", OUTPUT_PATH)

with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    print("\nSaved datasets:")
    for key in store.keys():
        print(" ", key)

Cleaned dataset saved successfully.
File: kaggle/working/iawe_fundamental_cleaned_6s.h5

Saved datasets:
  /house_1/air_conditioner_1
  /house_1/air_conditioner_2
  /house_1/fridge
  /house_1/iron
  /house_1/kitchen_outlets
  /house_1/laptop_computer
  /house_1/television
  /house_1/washing_machine
  /house_1/water_filter
  /house_1/water_motor


In [18]:
# ============================================================
# CELL 17 — VERIFY SAVED FILE
# ============================================================

import os

OUTPUT_PATH = "kaggle/working/iawe_fundamental_cleaned_6s.h5"

print("File exists:", os.path.exists(OUTPUT_PATH))

with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    print("\nSaved dataset structure:")
    for key in store.keys():
        df = store[key]
        print(f"{key}: {len(df):,} rows | columns = {list(df.columns)}")

File exists: True

Saved dataset structure:
/house_1/air_conditioner_1: 72,695 rows | columns = ['mains', 'appliance']
/house_1/air_conditioner_2: 91,051 rows | columns = ['mains', 'appliance']
/house_1/fridge: 810,940 rows | columns = ['mains', 'appliance']
/house_1/iron: 2,338 rows | columns = ['mains', 'appliance']
/house_1/kitchen_outlets: 3,133 rows | columns = ['mains', 'appliance']
/house_1/laptop_computer: 427,850 rows | columns = ['mains', 'appliance']
/house_1/television: 91,135 rows | columns = ['mains', 'appliance']
/house_1/washing_machine: 6,224 rows | columns = ['mains', 'appliance']
/house_1/water_filter: 336,322 rows | columns = ['mains', 'appliance']
/house_1/water_motor: 467,244 rows | columns = ['mains', 'appliance']


In [19]:
import h5py
with h5py.File('kaggle/working/iawe_fundamental_cleaned_6s.h5', 'r') as f:
    print(list(f.keys()))

['house_1']


In [20]:
import os
print(os.getcwd())
print(os.listdir('/kaggle/working'))

/kaggle/working
['kaggle', '__notebook__.ipynb']


In [21]:
import h5py
import pandas as pd

with h5py.File('kaggle/working/iawe_fundamental_cleaned_6s.h5', 'r') as f:
    print(f['house_1'])  # check if it's a Dataset or a Group
    print(dict(f['house_1'].attrs))  # any metadata attached?

<HDF5 group "/house_1" (10 members)>
{'CLASS': np.bytes_(b'GROUP'), 'TITLE': Empty(dtype=dtype('S1')), 'VERSION': np.bytes_(b'1.0')}


In [22]:
import h5py

with h5py.File('kaggle/working/iawe_fundamental_cleaned_6s.h5', 'r') as f:
    print(list(f['house_1'].keys()))

['air_conditioner_1', 'air_conditioner_2', 'fridge', 'iron', 'kitchen_outlets', 'laptop_computer', 'television', 'washing_machine', 'water_filter', 'water_motor']


In [23]:
import h5py

with h5py.File('kaggle/working/iawe_fundamental_cleaned_6s.h5', 'r') as f:
    print(list(f['house_1'].keys()))
    for k in f['house_1'].keys():
        print(k, f['house_1'][k])

['air_conditioner_1', 'air_conditioner_2', 'fridge', 'iron', 'kitchen_outlets', 'laptop_computer', 'television', 'washing_machine', 'water_filter', 'water_motor']
air_conditioner_1 <HDF5 group "/house_1/air_conditioner_1" (2 members)>
air_conditioner_2 <HDF5 group "/house_1/air_conditioner_2" (2 members)>
fridge <HDF5 group "/house_1/fridge" (2 members)>
iron <HDF5 group "/house_1/iron" (2 members)>
kitchen_outlets <HDF5 group "/house_1/kitchen_outlets" (2 members)>
laptop_computer <HDF5 group "/house_1/laptop_computer" (2 members)>
television <HDF5 group "/house_1/television" (2 members)>
washing_machine <HDF5 group "/house_1/washing_machine" (2 members)>
water_filter <HDF5 group "/house_1/water_filter" (2 members)>
water_motor <HDF5 group "/house_1/water_motor" (2 members)>


In [24]:
import h5py

with h5py.File('kaggle/working/iawe_fundamental_cleaned_6s.h5', 'r') as f:
    fridge = f['house_1/fridge']
    print(list(fridge.keys()))
    for k in fridge.keys():
        item = fridge[k]
        print(k, type(item))
        if isinstance(item, h5py.Dataset):
            print('  shape:', item.shape, 'dtype:', item.dtype)
            print('  sample:', item[:5])
        else:
            print('  sub-keys:', list(item.keys()))

['_i_table', 'table']
_i_table <class 'h5py._hl.group.Group'>
  sub-keys: ['appliance', 'index', 'mains']
table <class 'h5py._hl.dataset.Dataset'>
  shape: (810940,) dtype: [('index', '<i8'), ('mains', '<f4'), ('appliance', '<f4')]
  sample: [(1370563200000000000, 323.05338, 0.17449999)
 (1370563206000000000, 330.1709 , 0.19433333)
 (1370563212000000000, 330.06155, 0.141     )
 (1370563218000000000, 330.33618, 0.171     )
 (1370563224000000000, 329.98145, 0.13466667)]
